In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
import os

In [10]:
data_path = "CHOOSE_YOUR_FILE_PATH.csv"

df = pd.read_csv(data_path)

df = df.dropna(subset=['LATITUDE', 'LONGITUDE'])

df = df[(df["LONGITUDE"] >= -82) & (df["LONGITUDE"] <= -78) & (df["LATITUDE"] >= 38) & (df["LATITUDE"] <= 43)]

# Remove any accidental leading or trailing spaces from the operational complexity and environmental priority text
df["operational_complexity"] = df["operational_complexity"].str.strip()
df["environmental_priority"] = df["environmental_priority"].str.strip()

# Create a hash map that assigns each complexity a score oh how many days it can take to plug a well ranging from 1.5 - 7.0
complexity_days = {
    "High Complexity (Combination Hazard)": 7.0,
    "High Complexity (Multiple Wellbores)": 6.0,
    "High Complexity": 5.0,
    "Medium Complexity": 3.0,
    "Low Complexity": 1.5,
    "Unknown Physical Danger": 2.0,
}
# Create a hash map that assigns each complexity a score ranging from 1.5 - 4.0
priority_days = {
    "Tier 1: Extreme Danger": 4.0,
    "Tier 2: Extreme Danger": 4.0,
    "Tier 3: Moderate-High Danger": 2.5,
    "Tier 3: High Danger (Orphaned Injection)": 3.0,
    "Tier 3: High Danger (Orphaned Multi-Wellbore)": 3.0,
    "Tier 4: High Danger (Orphaned / Unknown Downhole Profile)": 2.5,
    "Tier 4: Moderate-High Danger (Abandoned Multi-Wellbore)": 2.5,
    "Tier 4: Moderate-High Danger (Abandoned Injection Well)": 2.5,
    "Tier 5: Moderate Danger": 1.5,
    "Tier 6: High Planning Priority (Orphaned Oil)": 1.0,
    "Tier 6: High Planning Priority": 1.0,
    "Tier 6: Moderate Planning Priority": 0.5,
    "Tier 7: Standard Risk": 0.2,
    "Tier 7: Standard Risk (Dry Hole)": 0.1,
}
#Create a complexity score  in order to map the complexity days based on their assigned scoring number.
df['complexity_score'] = df['operational_complexity'].map(complexity_days).fillna(0)

#Create a priority score in order to map the environmental priority based on their assigned scoring number.
df['priority_score'] = df['environmental_priority'].map(priority_days).fillna(0)

#Create 'noise' to replicate a real work environment where problems can occur and adds on to how long the field workers spend on a well plug.
field_noise = np.random.uniform(1,5,len(df))

#Calculate the final target variable by combining base scores and add field_noise.
df["ACTUAL_DAYS_REQUIRED"] = (df["complexity_score"] + df["priority_score"]) + field_noise

# Confirm successful calculation to the user
print("Your timeline targets are successfully calculated.")

# Separate the dataset into features (X) and the target variable (y)
X = df[["COUNTY", "operational_complexity", "environmental_priority"]]
y = df["ACTUAL_DAYS_REQUIRED"]

np.random.seed(42)

# Convert categorical text columns into numeric binary columns (One-Hot Encoding)
X_numerical = pd.get_dummies(X)

# Import the utility to split data for training and validation
from sklearn.model_selection import train_test_split

# Split data: 80% for training the model, 20% for testing its accuracy
X_train, X_test, y_train, y_test = train_test_split(X_numerical, y, test_size=0.2, random_state=42)

# Uses a repeatable 5-fold cross-validation strategy by randomly shuffling the dataset
cv_strategy = KFold(n_splits = 5, shuffle = True, random_state = 42)

#Alpha controls the penalty size for complex models, the algorithm adds a penalty if the model tries to overfit by making coefficients huge.
alphas = [0.0001,0.001,0.01, 0.1, 1.0, 10.0, 100.0]

# Initialize a Ridge Regression model and use alphas to evaluate then and select the best one and use cv to select the best alpha.
model = RidgeCV(alphas=alphas,fit_intercept=True, scoring='neg_mean_absolute_error',cv=cv_strategy)

# Train the model using the training data features and target labels
model.fit(X_train, y_train)

print(f"Best Alpha: {model.alpha_}")

# 3. See the final model coefficients (weights)
print(f"Coefficients: {model.coef_}")

# Use the trained Ridge model to predict the required days for the unseen test dataset
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:.2f} days")
print(f"R-squared Score: {r2:.2f}")

# 2. Set up a fresh figure
plt.figure(figsize=(8, 8))

# 3. Plot the true days vs the AI's guesses as small circular dots
plt.scatter(y_test, y_pred, color="darkorange", alpha=0.6, edgecolors="w", s=50)

# 4. Draw a perfect diagonal reference line (The "Perfect Guess" line)
perfect_line_range = [y_test.min(), y_test.max()]
plt.plot(
    perfect_line_range,
    perfect_line_range,
    color="navy",
    linestyle="--",
    linewidth=2,
    label="Perfect Predictions",
)

# 5. Add labels and legends
plt.title(
    "Model Evaluation: Actual vs. Predicted Plugging Timelines",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("True Days Required (With Real-World Noise)", fontsize=12)
plt.ylabel("AI Predicted Days Required", fontsize=12)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

# 6. Render the chart
plt.tight_layout()
plt.show()

Jupyter is currently looking inside: C:\Users\vasqu\PycharmProjects\PythonProject\ Hello World
Files found in this folder: ['.idea', '.venv', 'Calculator', 'DATA CLEANING TOP UEFA SCORERS', 'DATA CLEANING UEFA.ipynb', 'DBSCAN CLUSTERING.ipynb', 'df.ipynb', 'difficult.py', 'erfe.py', 'Estimated Days to Plug Based on Tiers.py', 'Fds.ipynb', 'game my version.py', 'Games.py', 'hello.py', 'K-CLUSTER 7 TIER MODEL.ipynb', 'K-CLUSTERS COORDINATES.ipynb', 'K-CLUSTERS PA WELLS MODEL.py', 'kcluster.ipynb', 'KCLUSTERS.ipynb', 'main.py', 'MEASURING PRIORITY FOR PA WELLS.py', 'MY NAT GAS.ipynb', 'My version.py', 'NAT GAS 2.ipynb', 'NAT GAS.ipynb', 'new.ipynb', 'number guessing game.py', 'pandas.ipynb', 'Pandas.py', 'Panthas practice.py', 'Password.py', 'password.txt', 'pg_final_analytics.csv', 'project.py', 'RIDGE REG.ipynb', 'RidgeCV Model.ipynb', 'Rock paper scissors.py', 'Simple Moving Average Signal Strat.ipynb', 'STOCKS.ipynb', 'UEFA.ipynb']


FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'